# S48_03 — ReAct and LangChain Agents

**ReAct** (Reasoning + Acting, Yao et al. 2022) is a prompting framework that interleaves reasoning traces with actions. The model explicitly thinks before acting, then observes results and reasons again.

```
Thought: I need to find the population of France
Action: search("France population 2024")
Observation: France has a population of approximately 68 million
Thought: Now I can answer the question
Answer: France's population is approximately 68 million
```

## LangChain agent with tools

In [ ]:
# pip install langchain langchain-anthropic langchain-community duckduckgo-search
from langchain_anthropic import ChatAnthropic
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool
import math

# Define tools with @tool decorator
@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Use Python syntax. Example: '2 ** 10' or 'sqrt(16)'"""
    safe_globals = {'__builtins__': {}, 'sqrt': math.sqrt, 'pi': math.pi, 'e': math.e}
    try:
        return str(eval(expression, safe_globals))
    except Exception as e:
        return f'Error: {e}'

@tool
def lookup_country(country: str) -> str:
    """Look up basic facts about a country: population, capital, area."""
    data = {
        'france': {'capital': 'Paris', 'population': '68M', 'area_km2': 551695},
        'germany': {'capital': 'Berlin', 'population': '84M', 'area_km2': 357114},
        'japan': {'capital': 'Tokyo', 'population': '125M', 'area_km2': 377975},
    }
    return str(data.get(country.lower(), f'No data for {country}'))

tools = [calculate, lookup_country]

print('Tools registered:', [t.name for t in tools])

In [ ]:
# Build agent
llm = ChatAnthropic(model='claude-haiku-4-5-20251001', temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant with access to tools. Use them when needed.'),
    ('human', '{input}'),
    ('placeholder', '{agent_scratchpad}'),
])

agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=5)

# Run — will show Thought/Action/Observation trace
result = executor.invoke({
    'input': 'What is the combined population of France and Germany? Give the answer in millions.'
})
print('\nFinal answer:', result['output'])

## Built-in LangChain tools

In [ ]:
# LangChain provides many ready-made tools
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun

# Web search
# search = DuckDuckGoSearchRun()
# result = search.run('latest Python version 2024')

# Wikipedia
# wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
# result = wiki.run('transformer neural network')

# Other useful tools:
# - ArxivQueryRun: search academic papers
# - PythonREPLTool: execute Python code
# - FileManagementToolkit: read/write files
# - SQLDatabaseToolkit: query SQL databases
# - RequestsGetTool: make HTTP requests

print('Common LangChain tools:')
print('  DuckDuckGoSearchRun — web search (no API key)')
print('  WikipediaQueryRun — Wikipedia search')
print('  PythonREPLTool — execute Python (sandbox carefully!)')
print('  SQLDatabaseToolkit — natural language to SQL')
print('  ArxivQueryRun — academic paper search')

## Agent memory — remembering across turns

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Session-based memory
store = {}  # {session_id: history}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

prompt_with_history = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    ('placeholder', '{chat_history}'),
    ('human', '{input}'),
    ('placeholder', '{agent_scratchpad}'),
])

agent_with_history = RunnableWithMessageHistory(
    AgentExecutor(agent=create_tool_calling_agent(llm, tools, prompt_with_history), tools=tools),
    get_session_history,
    input_messages_key='input',
    history_messages_key='chat_history',
)

session = {'configurable': {'session_id': 'user123'}}

r1 = agent_with_history.invoke({'input': 'What is the population of Japan?'}, config=session)
print('Turn 1:', r1['output'])

r2 = agent_with_history.invoke({'input': 'What is twice that number?'}, config=session)
print('Turn 2:', r2['output'])  # agent remembers Japan's population

## ReAct vs tool calling

| | ReAct (text-based) | Native tool calling |
|-|-------------------|-----------------------|
| How it works | Prompts model to output "Thought/Action" text | Model outputs structured JSON tool call |
| Reliability | Depends on prompt; can hallucinate format | Enforced by the API |
| Parsing | Custom string parsing | Automatic (SDK handles) |
| Usage today | Legacy; mostly replaced | Current standard |

Modern LangChain (`create_tool_calling_agent`) uses native tool calling under the hood. ReAct-style prompting is mainly useful when tool calling is not available.

Next: [S48_04_langgraph.ipynb](./S48_04_langgraph.ipynb)